# 🌿 XAI Computational Efficiency Benchmarking
### Evaluating Saliency, Grad-CAM, LIME & SHAP on Lightweight Edge Models

This notebook clones the project from GitHub, installs all dependencies
(including the **Kaggle CLI**), configures credentials, downloads the
**real PlantVillage Tomato dataset**, and runs the full CRISP-DM pipeline.

> **Tip:** Go to `Runtime → Change runtime type → T4 GPU` for much faster SHAP & LIME runs.

---
**What this notebook does:**
1. ⚙️ Sets up the environment (clone repo + install packages incl. `kaggle`)
2. 🔑 Configures Kaggle credentials (upload or use Drive-stored token)
3. 📥 Downloads the real PlantVillage Tomato leaf images via Kaggle API
4. 🏋️ Fine-tunes MobileNetV3 / ShuffleNetV2 on real data
5. 🔍 Generates visual explanation maps (Saliency, Grad-CAM, LIME, SHAP)
6. 📊 Benchmarks each XAI method over **N runs** and plots the distributions

## Step 1 — Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/bonsoirval/explainability_impact"
REPO_DIR = "explainability_impact"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 2 — Install dependencies (including Kaggle)

Installs PyTorch, Captum, LIME, SHAP, **and the Kaggle CLI** from `requirements.txt`.  
First run takes ~2–3 minutes.

In [ ]:
# Install all project requirements (kaggle>=1.6.0 is included)
!pip install -q -r requirements.txt

# Verify the Kaggle CLI is available
!kaggle --version
print("✅ All dependencies installed (Kaggle CLI ready).")

## Step 2b — Configure Kaggle credentials

The pipeline needs a valid `kaggle.json` API token to download the dataset.  
**Three options — the cell picks the first one that works:**

| Option | When to use |
|--------|-------------|
| **A** | You already mounted Google Drive and have `kaggle.json` stored there |
| **B** | You want to upload the file manually right now |
| **C** | `~/.kaggle/kaggle.json` is already set (e.g. a re-run in the same session) |

Get a token at [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token**.

In [ ]:
import os, shutil

KAGGLE_DIR  = os.path.expanduser("~/.kaggle")
KAGGLE_JSON = os.path.join(KAGGLE_DIR, "kaggle.json")

# ── Option A: load from Google Drive (edit path if needed) ────────────────
DRIVE_KAGGLE_JSON = "/content/drive/MyDrive/kaggle.json"   # ← change if stored elsewhere

def _install_kaggle_json(src: str) -> None:
    """Copy src → ~/.kaggle/kaggle.json with correct permissions."""
    os.makedirs(KAGGLE_DIR, exist_ok=True)
    shutil.copy(src, KAGGLE_JSON)
    os.chmod(KAGGLE_JSON, 0o600)
    print(f"✅ kaggle.json installed from: {src}")

if os.path.exists(KAGGLE_JSON):
    # Option C: already present (re-run)
    print("✅ kaggle.json already present — no action needed.")

elif os.path.exists(DRIVE_KAGGLE_JSON):
    # Option A: from Google Drive
    _install_kaggle_json(DRIVE_KAGGLE_JSON)

else:
    # Option B: manual upload
    print("kaggle.json not found. Please upload your Kaggle API token file:")
    from google.colab import files
    uploaded = files.upload()           # user picks kaggle.json
    if "kaggle.json" in uploaded:
        _install_kaggle_json("kaggle.json")
    else:
        raise FileNotFoundError(
            "No file named 'kaggle.json' was uploaded. "
            "Please download it from https://www.kaggle.com/settings → API."
        )

# ── Quick sanity check ────────────────────────────────────────────────────
!kaggle datasets list --max-size 1 -q 2>&1 | head -3
print("\n✅ Kaggle authentication verified.")

## Step 2c — (Optional) Mount Google Drive for persistent storage

Mount Drive **before** running the pipeline if you want results and model weights
to survive session restarts.  Skip this cell if you don't need persistence.

In [ ]:
# Uncomment to mount Drive
# from google.colab import drive
# drive.mount('/content/drive')
# print("✅ Google Drive mounted.")

## Step 3 — Configuration

Adjust these variables to control the experiment:

| Variable | Meaning |
|---|---|
| `MODEL` | `mobilenet_v3` or `shufflenet_v2` |
| `RUNS` | How many times each XAI method is timed (10, 100, 500, 1000 …) |
| `EPOCHS` | Training epochs (1 is fine for benchmarking) |
| `METHODS` | Subset of XAI methods to run, or all four |
| `CROP` | Crop prefix to filter PlantVillage class folders |
| `IMAGE_TYPE` | `color` \| `segmented` \| `grayscale` |

In [ ]:
MODEL       = "mobilenet_v3"           # "mobilenet_v3" | "shufflenet_v2"
RUNS        = 10                       # change to 100 / 500 / 1000 as needed
EPOCHS      = 1
METHODS     = ["Saliency", "Grad-CAM", "LIME", "SHAP"]

# PlantVillage dataset settings
CROP        = "Tomato"                 # crop prefix to filter class folders
IMAGE_TYPE  = "color"                  # "color" | "segmented" | "grayscale"
KAGGLE_SLUG = "abdallahalidev/plantvillage-dataset"

RESULTS_DIR = "phase_5_evaluation/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Model        : {MODEL}")
print(f"Runs         : {RUNS}")
print(f"Epochs       : {EPOCHS}")
print(f"Methods      : {METHODS}")
print(f"Crop filter  : {CROP}")
print(f"Image type   : {IMAGE_TYPE}")
print(f"Kaggle slug  : {KAGGLE_SLUG}")

## Step 4 — Run the full CRISP-DM pipeline

This single cell runs all six phases end-to-end:
- **Phase 2** – Downloads real PlantVillage Tomato images via Kaggle (~1 GB for color Tomato classes)
- **Phase 3** – Preprocesses & splits data
- **Phase 4** – Fine-tunes the selected model
- **Phase 5** – Generates explanations + benchmarks XAI methods
- **Phase 6** – Exports ONNX model

With `RUNS=10` on a T4 GPU this takes **~10–20 minutes** (dataset download included).

In [ ]:
methods_arg = " ".join(METHODS)

!python run_pipeline.py \
    --model      {MODEL}       \
    --epochs     {EPOCHS}      \
    --runs       {RUNS}        \
    --methods    {methods_arg} \
    --crop       {CROP}        \
    --image_type {IMAGE_TYPE}  \
    --kaggle_slug {KAGGLE_SLUG}

## Step 5 — View the explanation maps

In [ ]:
from IPython.display import Image, display
import os

explanations_path = os.path.join(RESULTS_DIR, f"{MODEL}_explanations.png")
print("Explanation maps (Saliency | Grad-CAM | LIME | SHAP):")
display(Image(filename=explanations_path))

## Step 6 — View the benchmark distribution plots

Box plots + jittered individual run points for **Latency** and **Peak RAM** across all methods.

In [ ]:
dist_path = os.path.join(RESULTS_DIR, f"{MODEL}_benchmark_distributions.png")
print(f"Distribution plot ({RUNS} runs per method):")
display(Image(filename=dist_path))

## Step 7 — Inspect the aggregated benchmark table

In [ ]:
import pandas as pd

report_path = os.path.join(RESULTS_DIR, f"{MODEL}_benchmark_report.csv")
df = pd.read_csv(report_path)
print(f"\nAggregated benchmark summary ({RUNS} runs):")
df.style.format({
    'Latency Mean (s)'   : '{:.4f}',
    'Latency Std (s)'    : '{:.4f}',
    'Peak RAM Mean (MB)' : '{:.2f}',
    'Peak VRAM Mean (MB)': '{:.2f}',
}).background_gradient(subset=['Latency Mean (s)'], cmap='YlOrRd')

## Step 8 — Inspect every individual run

The raw run-by-run CSV lets you plot your own graphs or inspect variance.

In [ ]:
raw_path = os.path.join(RESULTS_DIR, f"{MODEL}_raw_runs.csv")
raw_df = pd.read_csv(raw_path)
print(f"Total individual runs recorded: {len(raw_df)}")
raw_df

## Step 9 — Dataset class distribution (Phase 2 sanity check)

Verifies whether the PlantVillage Tomato subset is class-balanced before training.
Imbalanced classes require stratified sampling or weighted loss — document this in your thesis.

In [ ]:
from IPython.display import Image, display
import os

display(Image(filename=os.path.join(RESULTS_DIR, 'dataset_class_distribution.png')))

## Step 10 — Per-class Precision / Recall / F1 (Phase 4 evaluation)

Grouped bar chart showing how well the model handles each individual disease class.
Essential for detecting systematic failures on minority or visually similar classes.

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, f'{MODEL}_per_class_metrics.png')))

## Step 11 — Violin latency plot (richer distribution view)

A violin plot surfaces bimodality, heavy tails, and outlier runs that a box plot masks.
Report this alongside the standard box plot in your thesis results chapter.

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, f'{MODEL}_violin_latency.png')))

## Step 12 — Throughput (samples / second)

Answers the edge-deployment question: *how many leaf images can be explained per second?*  
Error bars show run-to-run stability.

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, f'{MODEL}_throughput.png')))

## Step 13 — Computational efficiency ratio (RAM × Latency)

Collapses memory and speed into a single *MB·s* footprint metric per method.  
Lower is better — ideal for comparing methods when both axes matter.

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, f'{MODEL}_ram_efficiency.png')))

## Step 14 — Radar chart (multi-dimensional cost overview)

Summarises Latency, RAM, VRAM, and stability (CV) per method on a single spider plot.  
Useful as a single *at-a-glance* figure in the thesis conclusion chapter.

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, f'{MODEL}_method_radar.png')))

## Step 15 — Cumulative time budget

Shows total wall-clock cost as a function of number of XAI calls.  
Directly answers: *'Is this method feasible for 1000-image field deployment?'*

In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, f'{MODEL}_cumulative_time.png')))

## (Optional) Step 16 — Cross-model comparison

Run **ShuffleNetV2** using Step 17 below, then display side-by-side comparisons of  
**MobileNetV3 vs ShuffleNetV2** for both mean latency and peak RAM.

In [ ]:
import os
lat_cmp  = os.path.join(RESULTS_DIR, 'cross_model_latency_comparison.png')
ram_cmp  = os.path.join(RESULTS_DIR, 'cross_model_ram_comparison.png')

if os.path.exists(lat_cmp):
    print('Latency comparison:')
    display(Image(filename=lat_cmp))
if os.path.exists(ram_cmp):
    print('RAM comparison:')
    display(Image(filename=ram_cmp))
if not os.path.exists(lat_cmp):
    print('Run ShuffleNetV2 first (Step 17) to generate cross-model comparisons.')

## (Optional) Step 17 — Run ShuffleNetV2 for cross-model comparison

Uncomment and run to benchmark ShuffleNetV2, then re-run Step 16.

In [ ]:
# MODEL2       = 'shufflenet_v2'
# methods_arg  = ' '.join(METHODS)
#
# !python run_pipeline.py \\
#     --model      {MODEL2}       \\
#     --epochs     {EPOCHS}       \\
#     --runs       {RUNS}         \\
#     --methods    {methods_arg}  \\
#     --crop       {CROP}         \\
#     --image_type {IMAGE_TYPE}   \\
#     --kaggle_slug {KAGGLE_SLUG}

---
## Complete output files reference

| File | Phase | Description |
|------|-------|-------------|
| `dataset_class_distribution.png` | 2 | Per-class image counts — imbalance check |
| `{model}_training_curves.png` | 4 | Train/val loss & accuracy |
| `{model}_confusion_matrix.png` | 4 | Row-normalised confusion matrix |
| `{model}_per_class_metrics.png` | 4 | Precision / Recall / F1 per class |
| `{model}_confidence_bar.png` | 5 | Softmax confidence for the explained sample |
| `{model}_explanations.png` | 5 | Saliency / Grad-CAM / LIME / SHAP side-by-side |
| `{model}_multi_class_explanations.png` | 5 | Per-disease explanation grid |
| `{model}_benchmark_distributions.png` | 5 | Box + jitter: latency & RAM |
| `{model}_violin_latency.png` | 5 | Violin plot — full latency distribution |
| `{model}_latency_vs_ram.png` | 5 | 2-D cost scatter (latency × RAM) |
| `{model}_throughput.png` | 5 | Samples/second throughput |
| `{model}_ram_efficiency.png` | 5 | RAM × Latency efficiency ratio |
| `{model}_method_radar.png` | 5 | Spider chart — 4-metric overview |
| `{model}_run_over_run.png` | 5 | Latency per run (warmup / variance) |
| `{model}_cumulative_time.png` | 5 | Cumulative time budget over N runs |
| `{model}_cv_stability.png` | 5 | Coefficient of variation stability |
| `{model}_benchmark_report.csv` | 5 | Aggregated mean ± std stats |
| `{model}_raw_runs.csv` | 5 | Every individual run data point |
| `cross_model_latency_comparison.png` | 5 | MobileNetV3 vs ShuffleNetV2 — latency |
| `cross_model_ram_comparison.png` | 5 | MobileNetV3 vs ShuffleNetV2 — RAM |
| `phase_6_deployment/{model}.onnx` | 6 | ONNX model for edge deployment |